In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_DIR  = '/kaggle/input/competitions/datathon-2026-round-1/'

print("Đang load dữ liệu...")
orders      = pd.read_csv(DATA_DIR + "orders.csv",      parse_dates=["order_date"])
products    = pd.read_csv(DATA_DIR + "products.csv")
returns     = pd.read_csv(DATA_DIR + "returns.csv",     parse_dates=["return_date"])
web_traffic = pd.read_csv(DATA_DIR + "web_traffic.csv", parse_dates=["date"])
order_items = pd.read_csv(DATA_DIR + "order_items.csv", low_memory=False)
customers  = pd.read_csv(DATA_DIR + 'customers.csv')
geography  = pd.read_csv(DATA_DIR + 'geography.csv')
inventory  = pd.read_csv(DATA_DIR + 'inventory.csv')
payments  = pd.read_csv(DATA_DIR + 'payments.csv')
reviews    = pd.read_csv(DATA_DIR + 'reviews.csv')
shipments  = pd.read_csv(DATA_DIR + 'shipments.csv')
promotions = pd.read_csv(DATA_DIR + 'promotions.csv')
sales  = pd.read_csv(DATA_DIR + 'sales.csv')
sample_submission  = pd.read_csv(DATA_DIR + 'sample_submission.csv')
print("Load xong!\n")

# Input data files are available in the read-only "../input/" directory
# Running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Q1


multi_order_customers = (
    orders.groupby("customer_id")
    .filter(lambda x: len(x) > 1)
)
multi_order_customers = multi_order_customers.sort_values(
    ["customer_id", "order_date"]
)
gaps = (
    multi_order_customers
    .groupby("customer_id")["order_date"]
    .diff()          # khoảng cách giữa hai đơn liên tiếp
    .dt.days         # chuyển sang số ngày
    .dropna()        # bỏ dòng NaN (dòng đầu tiên của mỗi khách)
)

median_gap = gaps.median()
 
print(f"  Số khách hàng có >1 đơn  : {multi_order_customers['customer_id'].nunique():,}")
print(f"  Tổng số khoảng cách       : {len(gaps):,}")
print(f"  Trung vị inter-order gap  : {median_gap:.1f} ngày")

print()

In [ ]:
# Q2


products["gross_margin"] = (products["price"] - products["cogs"]) / products["price"]
margin_by_segment = (
    products
    .groupby("segment")["gross_margin"]
    .mean()
    .sort_values(ascending=False)
    .rename("Avg Gross Margin")
)

print("Gross margin trung bình theo Segment:")
for seg, val in margin_by_segment.items():
    marker = " => CAO NHẤT" if seg == margin_by_segment.idxmax() else ""
    print(f"    {seg:<15} {val:.4f}  ({val*100:.2f}%){marker}")
print()
print()

In [ ]:
# Q3



streetwear_ids = products.loc[
    products["category"] == "Streetwear", "product_id"
]
streetwear_returns = returns[returns["product_id"].isin(streetwear_ids)]

reason_counts = streetwear_returns["return_reason"].value_counts()

print()
print("  Lý do trả hàng (sắp xếp từ nhiều nhất đến ít nhất):")
for reason, count in reason_counts.items():
    pct    = count / len(streetwear_returns) * 100
    marker = " <= PHỔ BIẾN NHẤT" if reason == reason_counts.idxmax() else ""
    print(f"    {reason:<20} {count:>6,}  ({pct:.1f}%){marker}")
print()
print()

In [ ]:
# Q4
# bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

avg_bounce = (
    web_traffic
    .groupby("traffic_source")["bounce_rate"]
    .mean()
    .sort_values()
    .rename("Avg Bounce Rate")
)
print("  Bounce rate trung bình theo nguồn:")
for src, val in avg_bounce.items():
    marker = " => THẤP NHẤT" if src == avg_bounce.idxmin() else ""
    print(f"    {src:<20} {val:.6f}{marker}")
print()
print()

In [ ]:
# Q5

total_rows  = len(order_items)
promo_rows  = order_items["promo_id"].notna().sum()
pct_promo   = promo_rows / total_rows * 100

print(f"  Tổng số dòng trong order_items : {total_rows:>10,}")
print(f"  Dòng có promo_id (không null)  : {promo_rows:>10,}")
print(f"  Dòng không có promo_id (null)  : {total_rows - promo_rows:>10,}")
print(f"  Tỷ lệ có khuyến mãi           : {pct_promo:>10.2f}%")
print()
print()

In [ ]:
# Q6
merge_df = pd.merge(customers.dropna(subset=['age_group']), orders, on='customer_id', how = 'inner')

result_6 = (
    merge_df
    .groupby('age_group')
    .agg(total_orders=('order_id', 'count'),
         unique_customers=('customer_id', 'nunique')
    )
    .reset_index()
)

result_6['avg_order_per_customer'] = result_6['total_orders'] / result_6['unique_customers']
result_6 = result_6[['age_group', 'avg_order_per_customer']].sort_values('avg_order_per_customer')
print(result_6)


In [ ]:
# Q7
merge_df = (order_items.merge(orders, on='order_id', how = 'inner').merge(geography, on='zip', how='inner'))

merge_df['revenue'] = (merge_df['quantity'] * merge_df['unit_price']) - merge_df['discount_amount']

result_7 = (
     merge_df
    .groupby('region')['revenue']
    .sum()
    .round(2)
    .reset_index(name='total_revenue')
    .sort_values('total_revenue', ascending=False)
)
print(result_7)


In [ ]:
# Q8
result_8 = (
    orders[orders['order_status'] == 'cancelled']
    .groupby('payment_method')
    .size()
    .reset_index(name='buy')
    .sort_values('buy', ascending=False)
)
print(result_8)


In [ ]:
# Q9
merge = (order_items.merge(products, on='product_id',how='inner')
         .merge(returns, on=['order_id','product_id'],how = 'left', indicator=True))
merge['is_returned'] = (merge['_merge'] == 'both').astype(int)

result_9 = (
    merge
    .groupby('size')
    .agg(total_returns=('is_returned', 'sum'),total_items=('order_id', 'count'))
    .reset_index()
)

result_9['return_rate'] = result_9['total_returns'] / result_9['total_items']
result_9 = result_9[['size', 'return_rate']].sort_values('return_rate', ascending=False)
print(result_9)


In [ ]:
# Q10
result_10 = (
    payments
    .groupby('installments')['payment_value']
    .mean()
    .reset_index(name = 'avg_payment')
    .sort_values('avg_payment', ascending=False)
)
print(result_10)
